# Module 7 capstone, toolset twin: a line twin calibrated from the fab's log

**Digital Twins for Semiconductor Manufacturing · Module 7 · about 2 hours** · notebook version 2026-10-25

This is the **toolset twin** track of the capstone. You join prepared components into one twin of the furnace, lithography and etch line of Module 6, and test it the way a twin should be tested before anyone relies on it:

- the **SimPy line model** of Module 6, whose furnace tubes drift and are calibrated by the predictive rule of Modules 5 and 6;
- the fab's **lot log**: when each lot was released, and when it started and finished at each tool group.

The twin computes the **weekly mean cycle time** of the line from the week's lot releases, for capacity planning. This capstone validates it **retrospectively**: the twin replays past weeks with the release times the fab actually used (a trace-driven replay), so the test shows how well the line model reproduces the cycle time when the releases are known. It does not show how well the twin would forecast future weeks whose releases are not yet known: that test would drive the twin with the release plan available at the forecast date, and calibrate it only on data available by then. You estimate its settings from the log (the connection between the log and the model), write one function that turns a test result into a verdict, and fill in the **twin data sheet**: what the twin is for, what it uses, how it performed, where it must not be used, one risk, and what to do when confidence is low.

The "fab" here is the same simulator with settings of its own, which the twin has to estimate from the log, and with one thing the twin does not model. The test plan is fixed before any test runs:

| test | data | pass when |
|---|---|---|
| calibration | weeks 1 to 6 of the log | (estimates only) |
| held-out | weeks 7 to 30: the twin's replay of each week, with the week's actual releases, against the log | mean absolute weekly error (MAE) at most 1.0 h |
| baseline | the mean weekly cycle time of weeks 1 to 6, on the same weeks | (for comparison) |
| challenging | weeks 31 to 40, when the lithography tool breaks down now and then | MAE at most 1.0 h |

Steps 1 to 8 use the **course example line (KEY 2026)**, so you can compare your numbers with the Module 7 page. Step 9 repeats the tests on your own line, with the KEY from Question 1 of the **Module 7 project quiz for the toolset twin**. All data are **simulated**. The twin runs 20 replications for every test, so Steps 5, 8 and 9 take a while (up to about a minute each in the browser). Run each code cell with **Shift + Enter**, in order.

## Step 0: start Python

The first time, Python can take up to a minute to start. The cell prints the versions of the libraries and of the course files next to this notebook, including SimPy, which `line.py` imports from the wheel file next to it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy             # the course files use SciPy: importing it here loads it in the browser
import line, capstone, toolset_twin
from capstone import mae_ci, blank_datasheet, show_options, show_datasheet
from toolset_twin import (fab_log, downtime_log, twin_config, twin_run, weekly_cycle_time, replay, plot_weeks, week_window,
                          TOLERANCE_H, CAL_WEEKS, HELDOUT_WEEKS, CHALLENGING_WEEKS, WEEK_H)

EXAMPLE_KEY = 2026
TOLERANCE = TOLERANCE_H
print("Notebook version 2026-10-25")
print("NumPy", np.__version__, "| pandas", pd.__version__, "| SimPy", line.simpy.__version__, "| line.py", line.__version__,
      "| capstone.py", capstone.__version__, "| toolset_twin.py", toolset_twin.__version__)

## Step 1: the prepared components

A twin in a fab is rarely written from scratch: it is assembled from models and tools that were each built and tested on their own. This track supplies these components:

| component | from | what it does |
|---|---|---|
| `fab_log(key)`, `downtime_log(key)` | new data | 40 weeks of the line's lot log, and the lithography breakdowns the fab recorded |
| `twin_config(estimates)` | Module 6 | the twin's line settings from your estimates, with the predictive maintenance rule |
| `twin_run(cfg, key, weeks)`, `replay(cfg, key, weeks)` | Module 6 | the twin driven by the log's own release times: one replication, or the weekly mean over 20 |
| `weekly_cycle_time(log, weeks)` | Module 6 | the mean cycle time of the lots that leave in each week |
| `mae_ci(errors)` | Module 6 | the mean absolute error and the half-width of its 95 % interval |

The twin is **trace-driven**: it does not invent its own lot releases, it releases its lots at the times the fab released them. Its process times, tube drift and calibrations are its own and random, so 20 replications give its result for a week. That way a busy week in the fab is a busy week in the twin, and the comparison tests the model, not the luck of the releases (Law 2015 recommends this for validating a simulation against a real system). The price is that a replay is conditional on the releases that actually happened: it is a retrospective test of the line model, not a forecast.

What is missing is the **connection** between the log and the model: the twin's release rate and process times have to be estimated from the log. That is your part, in Step 3.

## Step 2: the fab's log

The log starts with the line already running. Each row is one lot: its release `t_in`, the start and end of its furnace run (`f_start`, `f_end`), of its lithography step (`l_start`, `l_end`), the start of its etch step `e_start`, and when it left the line `t_out`, all in hours. A lot still in the line when the log ends has NaN for the steps it has not reached. The downtime log lists the lithography breakdowns the fab recorded.

In [ ]:
log = fab_log(EXAMPLE_KEY)
down = downtime_log(EXAMPLE_KEY)
print(log.head(6).round(2).to_string(index=False))
print(f"{len(log)} lots in 40 weeks")
weeks_down = sorted(set((down["t_start"] // WEEK_H + 1).astype(int)))
print("lithography breakdowns recorded:", len(down), "in weeks", weeks_down[0] if weeks_down else "-", "to", weeks_down[-1] if weeks_down else "-")
observed_all = weekly_cycle_time(log, (1, 40))
print(observed_all.round(2).to_string())

## Step 3: your task, estimate the twin's settings from the log

The twin needs seven numbers, estimated from the lots released in the calibration weeks:

- `arrival_rate`: the lots released in the calibration window divided by its length in hours, `len(calib) / hours` (count every lot released, finished or not);
- `furnace_h`, `litho_h`, `etch_h`: the mean **process time** of each tool group: `f_end - f_start`, `l_end - l_start` and `t_out - e_start`. These start when the tool starts, so they leave out the waiting;
- `furnace_cv`, `litho_cv`, `etch_cv`: the coefficient of variation of each process time, its standard deviation divided by its mean. Use the sample standard deviation, n − 1 in the divisor, which is what pandas' `.std()` gives.

Lots still in the line have NaN for the steps they have not finished: drop them from each process time with `.dropna()` before taking the mean and the standard deviation.

**Your task:** write `twin_estimates(calib, hours)`. `calib` is the part of the log released in the calibration window and `hours` the window's length. Return a dict with the seven keys above.

Two notes on what this calibration uses. The log is read after week 40, so the lots released late in week 6 bring the steps they finish early in week 7 (7 lots for the example line); a forecast made at the end of week 6 could not use those steps. And the trace-driven replay takes its arrivals from the log's release times, so the release rate you estimate does not change the replay: it records the release rate at which the twin was validated, which is what a planner would need, with a release plan, to run the twin ahead.

In [ ]:
def twin_estimates(calib, hours):
    """The twin's release rate (lots/h), mean process times (h) and their coefficients of variation, from the lots released
    in the calibration window."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete twin_estimates() first.")

In [ ]:
a, z = week_window(CAL_WEEKS)
calib = log[(log["t_in"] >= a) & (log["t_in"] < z)]
est = twin_estimates(calib, z - a)
print({k: round(float(v), 4) for k, v in est.items()})
print("(the release rate should be near 0.7 to 0.76 lots per hour, the process times near 1.8, 1.0 and 1.2 h)")
cfg = twin_config(est)

## Step 4: verify the connected twin

**Verification and validation are different questions.** Verification asks whether the code computes what the model says. Validation asks whether the model's answers are close enough to the real system for the use you intend. (These are the definitions the National Academies use for digital twins, from the National Research Council's 2012 report.) Verification comes first: a validation result means little if the code does not do what you think it does. The tests below compare the connected twin with answers you can work out exactly.

Four tests on one replication of the twin for the held-out weeks: (1) it released exactly the lots the log released in those weeks (the trace-driven input works); (2) every lot's steps happen in order: furnace after release, lithography after the furnace, etch after lithography; (3) its mean lithography process time matches your estimate to within a few percent, as it must with about 4,000 lots drawn from it; (4) its mean cycle time is longer than the sum of the three process times, since lots also wait.

In [ ]:
one = twin_run(cfg, EXAMPLE_KEY, HELDOUT_WEEKS, seed=1)
a7, z30 = week_window(HELDOUT_WEEKS)
lots = one.lots
n_log = int(((log["t_in"] >= a7 - 200.0) & (log["t_in"] < z30)).sum())
ok1 = len(lots) == n_log
done = lots.dropna()
ok2 = bool(((done["f_start"] >= done["t_in"]) & (done["l_start"] >= done["f_end"]) & (done["e_start"] >= done["l_end"]) & (done["t_out"] >= done["e_start"])).all())
litho = (lots["l_end"] - lots["l_start"]).dropna()
ok3 = abs(litho.mean() / est["litho_h"] - 1.0) < 0.05
ok4 = (lots["t_out"] - lots["t_in"]).dropna().mean() > est["furnace_h"] + est["litho_h"] + est["etch_h"]
for i, ok in enumerate((ok1, ok2, ok3, ok4), 1):
    print(f"verification test {i}:", "ok" if ok else "FAILED: check Step 3")

## Step 5: validate on held-out weeks

Now the twin replays weeks 7 to 30, which played no part in the calibration. For each week, `replay()` runs 20 replications of the trace-driven twin and averages their weekly mean cycle time; `weekly_cycle_time()` gives what the log measured. The test unit is the **week**: `mae_ci()` gives the mean absolute weekly error with the half-width of its 95 % interval, t(0.975, n − 1) × s / √n, as in Module 6. The interval is approximate: it treats the weeks as independent, but lots still in the line at the end of a week carry its congestion into the next, so neighbouring weeks can be correlated.

In the plot, the shaded band shows the size of the tolerance, for scale. The verdict compares the MAE of all the weeks, printed above the lower panel with its interval, with the tolerance: a single week outside the band does not fail the twin, and an MAE within the tolerance does not make every week acceptable. This cell takes a while.

In [ ]:
observed = weekly_cycle_time(log, HELDOUT_WEEKS)
twin = replay(cfg, EXAMPLE_KEY, HELDOUT_WEEKS)
baseline = float(weekly_cycle_time(log, CAL_WEEKS).mean())
plot_weeks(observed, twin, baseline, title=f"Toolset twin, example line (KEY {EXAMPLE_KEY}), held-out weeks", window_name="held-out")
h_mae, h_hw, h_n = mae_ci(observed - twin)
print(f"held-out: MAE {h_mae:.3f} h, 95 % interval +/- {h_hw:.3f} h, over {h_n} weeks (tolerance {TOLERANCE} h)")

## Step 6: your task, `verdict()`

A test gives an error with an interval, not a single number, and the **tolerance** was set before the test, from the twin's intended use: here the mean absolute error must be at most **1 h**. Compare the whole interval with the tolerance:

- **meets**: the whole interval lies at or below the tolerance, MAE + half-width ≤ tolerance;
- **misses**: the whole interval lies above the tolerance, MAE − half-width > tolerance;
- **insufficient evidence**: the interval contains the tolerance, so this test cannot tell. More test data would narrow the interval.

"Insufficient evidence" is an honest result, not a failure of the method. It is better than a "meets" that the data do not support.

The 95 % interval from `mae_ci()` is **approximate**. It treats the test units (runs or weeks) as independent, and neighbouring units can be correlated: when an error carries over from one unit to the next, the true uncertainty is larger than the computed interval, most of all after a change, when the errors move together for several units. Read a verdict close to the tolerance with that in mind.

**Your task:** write `verdict(mae, half_width, tolerance)`. Return one of the three strings exactly as written here: `"meets"`, `"misses"` or `"insufficient evidence"`.

In [ ]:
def verdict(mae, half_width, tolerance):
    """"meets", "misses" or "insufficient evidence": the 95 % interval of the MAE, mae +/- half_width, against the tolerance."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete verdict() first.")

In [ ]:
print(verdict(0.30, 0.10, 0.5), "  (should be meets: the interval runs from 0.20 to 0.40)")
print(verdict(0.45, 0.10, 0.5), "  (should be insufficient evidence: 0.35 to 0.55 contains 0.5)")
print(verdict(0.90, 0.20, 0.5), "  (should be misses: 0.70 to 1.10 lies above 0.5)")
print()
print("held-out test of the example:", verdict(h_mae, h_hw, TOLERANCE))

## Step 7: compare with the baseline

A twin should beat the simplest thing that could work. The **baseline** here predicts every week as the mean weekly cycle time of the six calibration weeks: what a planner could do with the log alone and no model.

If the twin did not beat its baseline, that would be a valid finding too: the simpler method would be enough.

In [ ]:
b_mae, b_hw, b_n = mae_ci(observed - baseline)
print(f"baseline (mean of weeks 1 to 6, {baseline:.2f} h): MAE {b_mae:.3f} h, +/- {b_hw:.3f} h")
print(f"twin:                                  MAE {h_mae:.3f} h, +/- {h_hw:.3f} h")
print(f"the twin's error is {h_mae / b_mae:.0%} of the baseline's")

## Step 8: the challenging condition

Every validation needs one condition that is harder than normal operation. Here it is weeks 31 to 40, when the lithography tool, the bottleneck of Module 6, breaks down now and then and is repaired in a few hours. The twin has no breakdown model: nothing in the calibration weeks showed one. The challenge is built in on purpose, and the notebook says so: a twin is only trusted where it has been tested. This cell takes a while.

In [ ]:
obs_c = weekly_cycle_time(log, CHALLENGING_WEEKS)
twin_c = replay(cfg, EXAMPLE_KEY, CHALLENGING_WEEKS)
plot_weeks(obs_c, twin_c, baseline, title=f"Toolset twin, example line (KEY {EXAMPLE_KEY}), weeks with breakdowns", window_name="challenging")
c_mae, c_hw, c_n = mae_ci(obs_c - twin_c)
print(f"challenging: MAE {c_mae:.3f} h, +/- {c_hw:.3f} h, over {c_n} weeks:", verdict(c_mae, c_hw, TOLERANCE))

## Step 9: your own line

Open the **Module 7 project quiz for the toolset twin** in Canvas. Question 1 gives you a **KEY**, a whole number from 100 to 999. Type it below in place of `None` and run the cell. It repeats Steps 3 and 5 to 8 on **your line** with **your functions** and prints the numbers for the data sheet. It takes a minute or two.

In [ ]:
KEY = None      # <- your KEY from Question 1 of the project quiz, for example KEY = 417

if KEY is None:
    print("Type your KEY in place of None, then run this cell again.")
else:
    my_log = fab_log(KEY)
    a, z = week_window(CAL_WEEKS)
    my_cfg = twin_config(twin_estimates(my_log[(my_log["t_in"] >= a) & (my_log["t_in"] < z)], z - a))
    my_obs = weekly_cycle_time(my_log, HELDOUT_WEEKS)
    my_twin = replay(my_cfg, KEY, HELDOUT_WEEKS)
    my_base = float(weekly_cycle_time(my_log, CAL_WEEKS).mean())
    results = {"heldout": mae_ci(my_obs - my_twin),
               "challenging": mae_ci(weekly_cycle_time(my_log, CHALLENGING_WEEKS) - replay(my_cfg, KEY, CHALLENGING_WEEKS)),
               "baseline": mae_ci(my_obs - my_base)}
    print(f"Your line, KEY {KEY} (tolerance {TOLERANCE} h):")
    for name, (m, hw, n) in results.items():
        v = verdict(m, hw, TOLERANCE) if name != "baseline" else "(for comparison)"
        print(f"  {name:12s} MAE {m:.3f} h   +/- {hw:.3f}   n = {n:2d}   {v}")
    plot_weeks(my_obs, my_twin, my_base, title=f"Your line (KEY {KEY}), held-out weeks", window_name="held-out")

## Step 10: the twin data sheet

A twin that others will use needs a label, as a dataset or a model does (datasheets for datasets, Gebru et al. 2021; model cards, Mitchell et al. 2019). The **twin data sheet** has six fields. Fields 1, 2, 5 and 6 are fixed choices, field 3 holds your numbers from Step 9, and field 4 is a list of choices. The cell below prints the choices for this track.

How to fill in field 4, the conditions where the predictions should **not** be used:

- list every condition **outside** the range the twin was calibrated and tested in: nothing supports a claim there;
- list each tested condition whose verdict was **not** "meets" (that is, "misses" or "insufficient evidence");
- do **not** list a condition that your tests included and that met the tolerance: your results support it.

Field 5 needs a risk that this twin's data path really has, and a mitigation that reduces that risk (NIST IR 8356 discusses the security and trust risks of digital twins). Field 6 is what the user should do when the twin's confidence is low.

The check cell in Step 11 reads this data sheet and compares it with your results: the numbers must match Step 9 (3 decimals), the verdicts must follow your `verdict()` rule, and the choices must be consistent with what you found.

In [ ]:
show_options("toolset")

In [ ]:
datasheet = blank_datasheet()
datasheet["intended_use"] = None           # 1. one letter, for example "A"
datasheet["data_sources"] = None           # 2. one letter
datasheet["heldout_mae"] = None            # 3. your numbers from Step 9, 3 decimals
datasheet["heldout_verdict"] = None        #    "meets", "misses" or "insufficient evidence"
datasheet["challenging_mae"] = None
datasheet["challenging_verdict"] = None
datasheet["baseline_mae"] = None
datasheet["do_not_use"] = None             # 4. a list of letters, for example ["A", "D"]
datasheet["risk_mitigation"] = None        # 5. one letter
datasheet["low_confidence_action"] = None  # 6. one letter
datasheet["note"] = ""                     # optional, not checked

show_datasheet(datasheet, "toolset", results=globals().get("results"), tolerance=TOLERANCE, key=KEY, version="2026-10-25")

## Step 11: check your work

The cell below tests your functions on cases you cannot see, recomputes the validation results for your KEY with the course's reference functions, and compares your data sheet with them. If everything passes, it prints your **verification value**, a code made from your KEY and the three numbers of your data sheet. Type it into Question 1 of the **Module 7 project quiz for the toolset twin**.

If it says **NOT PASSED**, read the message: it tells you what to fix. Fix it, run the cells you changed, then run this cell again.

In [ ]:
# The course's check code is in dt_m7_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
import scipy, sklearn     # the check code uses SciPy and scikit-learn: importing them here loads them in the browser
from dt_m7_check import check_toolset_twin

passed, messages, value = check_toolset_twin(twin_estimates, verdict, datasheet, KEY)
for line_text in messages:
    print(line_text)
if passed:
    print()
    print(f"Verification value for KEY = {KEY}: {value:.3f}")
    print("Type this number into Question 1 of the Module 7 project quiz for the toolset twin.")

## What the data sheet says about the twin

You built a twin from parts that each worked on their own, verified that the connected code does what the model says, and validated it on data it had not seen, against a tolerance set before the test, next to a baseline and under one challenging condition. The data sheet records the result for whoever uses the twin next: what it is for, what it uses, how well it did, where it must not be used, one risk, and what to do when its confidence is low.

**What the toolset twin showed.** Calibrated from six weeks of the log and driven by each week's actual releases, the twin's replay reproduces the weekly cycle time to within its tolerance in normal weeks, better than the recent average. When the bottleneck starts to break down, the twin is wrong by hours, because breakdowns are not in it: that condition goes on the data sheet, and the twin needs a breakdown model, validated on new weeks, before it is used for those weeks. The test was retrospective, with the releases known. Before the twin plans future weeks from a release plan, it needs a prospective test of its own: the planned releases will differ from the actual ones, and the calibration may use only the data available at the forecast date.

**Validation is not done once.** The physical system keeps changing: thermocouples age, recipes change, tools are replaced. The National Academies' 2024 report on digital twins concludes that their verification, validation and uncertainty quantification must be a continual process, revisited as the physical system, the data and the use change. A data sheet carries a date for that reason.

**Claim only what the data support.** On simulated data, a result shows how well the twin recovers the course's simulator. It does not validate a twin of a real furnace or a real line.

To keep a copy of this notebook, choose **File > Download**.